In [1]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 4 — Challenge B: regression tree (R3)

**Scope:** P4.3, the rules of P4.4 (from the storage tree) and the Challenge B part of P4.5.

> **Team code style (delete this cell before submitting).** The team must be able to defend every line.
>
> - Prefer tools seen in class or named in the statement. If you cannot explain a line in one sentence, rewrite it.
> - Plain `for` loops and `if`s, one step per line, clear variable names. No lambdas, classes or long chains.
> - Same results, simpler code. After simplifying, restart, run all and check that the numbers did not change.
> - Interpretation goes in Markdown cells, in English. Check your notebook with `revisor_notebook.py`.
> - Every number written in a Markdown cell must appear in a printed output of the notebook.
>
> **Shared names, so that the notebooks can be joined:** `dev` (development table), `X`, `y` (cyber target), `z` (storage target), `latency_ns`, `NOMINAL`, `NUMERIC`, `FEATURES`, `make_preprocessor()`, `folds` (list of `(train_idx, val_idx)`, also passed as `cv=folds`), `CLASS_NAMES` (cyber).

## Setup from R1 and R4

Paste here the cells published by R1 and R4. They are **not** pasted into the final notebook.

In [2]:
# TODO: paste the data, preprocessing, folds and metrics cells
# TEMPORARY cell: what R1 defines for storage (it is not pasted into the final notebook)
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Our notebook is one folder deeper than R1's, so the paths start with ../../
STORAGE_DIR = "../../datos/storage"
FOLDS_DIR = "../../folds"

dev = pd.read_csv(STORAGE_DIR + "/storage_development.csv")
storage_folds = pd.read_csv(FOLDS_DIR + "/storage_folds.csv")
assert dev["row_id"].equals(storage_folds["row_id"]), "Fold rows are not aligned"

NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC

X = dev[FEATURES].copy()
latency_ns = dev["latency_ns"].copy()    # pandas Series, as in R1
z = np.log10(latency_ns)                 # also a pandas Series, as in R1

# Canonical folds, read from the course file (same code as R1)
folds = []
for fold_id in range(5):
    train_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() == fold_id)
    folds.append((train_idx, val_idx))

# Preprocessor (same as R1): full one-hot and numeric columns unchanged
def make_preprocessor():
    nominal_steps = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    preprocessor = ColumnTransformer([
        ("cat", nominal_steps, NOMINAL),
        ("num", "passthrough", NUMERIC),
    ])
    return preprocessor

# Only to print the cyber sizes used in the Markdown (in the final notebook R1's cyber cells print them)
cyber_dev = pd.read_csv("../../datos/cyber/cyber_development.csv")
cyber_groups = pd.read_csv("../../datos/cyber/cyber_groups.csv")
print("Cyber rows:", len(cyber_dev), "| cyber feature groups:", cyber_groups["feature_group"].nunique())

print("X:", X.shape, "| z:", z.shape)
for f in range(5):
    print("fold", f, "| training rows:", len(folds[f][0]), "| validation rows:", len(folds[f][1]))

Cyber rows: 34928 | cyber feature groups: 34883
X: (12190, 9) | z: (12190,)
fold 0 | training rows: 9750 | validation rows: 2440
fold 1 | training rows: 9750 | validation rows: 2440
fold 2 | training rows: 9750 | validation rows: 2440
fold 3 | training rows: 9755 | validation rows: 2435
fold 4 | training rows: 9755 | validation rows: 2435


# P4.3 — Tree regression

In [3]:
# TODO: imports for this part
import time     # to measure how long the search and each fit take
from sklearn.pipeline import Pipeline    # joins preprocessing and tree in one object that is fitted as a whole 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score   # MAE, MSE, RMSE and R2
from sklearn.model_selection import GridSearchCV    # tries every combination of the grid on the folds
from sklearn.tree import DecisionTreeRegressor    # the regression tree of this practice

In [4]:
# TODO: declare the grid values for this task
def make_tree():
    # A new, unfitted pipeline every time: each fold gets a fresh copy,
    # so nothing learned in one fold leaks into another
    return Pipeline([
        ("preprocessing", make_preprocessor()),  # one-hot nominal columns, pass numeric ones unchanges
        ("tree", DecisionTreeRegressor(random_state=PROJECT_RANDOM_STATE)),  # learns to predict z; seed fixed for ties
    ])

# Check 1: how many rows each configuration has
rows_per_config = dev.groupby(FEATURES).size()
print("Number of configurations:", len(rows_per_config))    # one entry per different configuration
print("Configurations by number of rows:")
print(rows_per_config.value_counts())          # how many configurations have 1 row, 2 rows...

# Check 2: in how many different folds each configuration appears (fold_id is only used for this check, never as a predictor)
check = dev[FEATURES].copy()        # a copy, so that dev itself is not changed
check["fold_id"] = storage_folds["fold_id"]                        # same row order as dev 
folds_per_config = check.groupby(FEATURES)["fold_id"].nunique()     # number of different folds per configuration
print("Configurations by number of different folds:")
print(folds_per_config.value_counts())      # if every configuration has 1, none is split between folds

# Declared grid: names start with "tree__" because the tree is the "tree" step of the pipeline
param_grid = {
    "tree__max_depth": [4, 6, 8, 10, 12, None],      # None = no depth limit
    "tree__min_samples_leaf": [5, 10, 20, 40],       # multiples of 5 replicates
    "tree__ccp_alpha": [0.0, 0.00001, 0.0001, 0.001],# pruning strength, in squared units of z
}

# Numbers used in the Markdown cells
# Leaf size in rows -> leaf size in configurations
for leaf in param_grid["tree__min_samples_leaf"]:
    configs = leaf // 5          # each configuration has 5 rows; // is integer division
    print("min_samples_leaf =", leaf, "-> at least", configs, "configurations per leaf")

# scale of z, to judge whether the ccp_alpha values are large or small
# ddof=0 divides by the number of rows, like the squared error that the tree minimises
print("Variance of z:", round(z.var(ddof=0), 4))     

# Size of the search
n_depth = len(param_grid["tree__max_depth"])     # number of max_depth values
n_leaf = len(param_grid["tree__min_samples_leaf"])       # number of min_samples_leaf values
n_alpha = len(param_grid["tree__ccp_alpha"])     # number of ccp_alpha values
n_combinations = n_depth * n_leaf * n_alpha      # every combination is tried
n_fits = n_combinations * len(folds)        # each combination is fitted once
print("Combinations:", n_combinations, "| fits:", n_fits)

Number of configurations: 2438
Configurations by number of rows:
5    2438
Name: count, dtype: int64
Configurations by number of different folds:
fold_id
1    2438
Name: count, dtype: int64
min_samples_leaf = 5 -> at least 1 configurations per leaf
min_samples_leaf = 10 -> at least 2 configurations per leaf
min_samples_leaf = 20 -> at least 4 configurations per leaf
min_samples_leaf = 40 -> at least 8 configurations per leaf
Variance of z: 0.7056
Combinations: 96 | fits: 480


**Replicates** 

The 12190 rows are 2438 configurations of the 9 predictors, each with exactly 5 rows, and every configuration appears in a single fold (both checks above). This has two consequences:
- A configuration is never in training and validation at the same time, so validation measures the error on configurations the tree has not seen.
- The 5 replicates of a configuration have identical predictors, so no split can separate them. Every leaf holds whole configurations, so at least 5 rows, and any `min_samples_leaf` from 1 to 5 gives the same tree. The fold 0 check below confirms it.

**Declared grid and why.**

- `max_depth` in {4, 6, 8, 10, 12, None}: from shallow trees that are easy to read to a tree with no depth limit (`None`), so the search shows where more depth stops helping on validation.
- `min_samples_leaf` in {5, 10, 20, 40}: at least 1, 2, 4 and 8 configurations per leaf. The value 5 stands for every value from 1 to 5, which all give the same tree; 40 forces leaves of at least 8 configurations.
- `ccp_alpha` in {0.0, 1e-05, 0.0001, 0.001}: in regression it is measured in squared units of z, whose variance is 0.7056. 0.0 means no pruning, and each non-zero value is ten times the previous one.
- 96 combinations, 480 fits on the shared canonical folds, scored with MAE(z), the primary metric of Challenge B.
- `random_state` is fixed because ties between equally good splits are broken with the estimator's random number generator, so the seed can change the fitted tree.

In [5]:
# Fold 0 only, to read the data before the search (nothing here chooses the hyperparameters)
train_idx = folds[0][0]   # positions of the training rows of fold 0
val_idx = folds[0][1]     # positions of the validation rows of fold 0

# 1) Unlimited tree: the defaults max_depth=None, min_samples_leaf=1, ccp_alpha=0: the tree grows as much as it can
full_tree = make_tree()
full_tree.fit(X.iloc[train_idx], z.iloc[train_idx])  # learn only from the training rows
# Error on the validation configurations, which the tree has never seen
full_mae = mean_absolute_error(z.iloc[val_idx], full_tree.predict(X.iloc[val_idx]))
# named_steps["tree"] is the tree inside the pipeline; get_n_leaves() counts its leaves
print("Unlimited tree, fold 0 | leaves:", full_tree.named_steps["tree"].get_n_leaves(),
     "| configurations in training:", len(train_idx) // 5,  # 5 rows per configuration
     "| validation MAE(z):", round(full_mae, 4))

# 2) Same tree with min_samples_leaf=5: replicates cannot be split, so we expect the same number of leaves
leaf5_tree = make_tree()
leaf5_tree.set_params(tree__min_samples_leaf=5)
leaf5_tree.fit(X.iloc[train_idx], z.iloc[train_idx])
print("min_sample_leaf = 5, fold 0 | leaves:", leaf5_tree.named_steps["tree"]. get_n_leaves())

# 3) Pruning path 
# cost_complexity_pruning_path belongs to the tree alone, so it needs the columns already transformed
# The preprocessor was fitted on these same rows only, inside full_tree
X_train_t = full_tree.named_steps["preprocessing"].transform(X.iloc[train_idx])
path = full_tree.named_steps["tree"].cost_complexity_pruning_path(X_train_t, z.iloc[train_idx])
alphas = path.ccp_alphas    # increasing: alphas[0] = 0 (no pruning), the last one prunes down to the root
print("Pruning path, fold 0 training rows | candidate alphas:", len(alphas),
     "| largest:", round(alphas[-1], 4), "(units of z squared)")

# For each declared value, count how many pruning steps of the path it applies:
# alphas[1:] skips the first value (0, no pruning), and "<= value" marks the steps that happen
for value in param_grid["tree__ccp_alpha"]:
    print("ccp_alpha =", value, "-> pruning steps of the path it applies:", int((alphas[1:] <= value).sum()))

Unlimited tree, fold 0 | leaves: 1950 | configurations in training: 1950 | validation MAE(z): 0.1263
min_sample_leaf = 5, fold 0 | leaves: 1950
Pruning path, fold 0 training rows | candidate alphas: 1836 | largest: 0.4316 (units of z squared)
ccp_alpha = 0.0 -> pruning steps of the path it applies: 0
ccp_alpha = 1e-05 -> pruning steps of the path it applies: 1365
ccp_alpha = 0.0001 -> pruning steps of the path it applies: 1671
ccp_alpha = 0.001 -> pruning steps of the path it applies: 1803


**Fold 0 check.** This cell only describes the data; nothing in it is used to choose the hyperparameters.

- Without any limit, the tree has 1950 leaves, exactly the 1950 configurations in the training rows of fold 0. It puts each training configuration in its own leaf and predicts the mean z of its replicates.
- With `min_samples_leaf` = 5 it has the same 1950 leaves, as expected from the replicates.
- The pruning path of this tree has 1836 candidate alphas; the largest, 0.4316, is the one that would prune it down to the root. The declared values apply 0, 1365, 1671 and 1803 pruning steps. Most steps already happen at very small alphas, which suggests that many neighbouring leaves have almost the same mean z, so merging them costs almost nothing. Even 0.001 is far below 0.4316, so the grid never comes close to a very small tree.
- The validation MAE(z) of the unlimited tree on fold 0 (0.1263) is shown only for reference.

In [6]:
# TODO: GridSearchCV(pipe, param_grid, cv=folds, scoring='neg_mean_absolute_error', return_train_score=True)
# Search object: it does not fit anything yet, it only stores what to try and hoy to score it
search = GridSearchCV(
    make_tree(),                             # the pipeline to tune; GridSearchCV fits a fresh copy each time
    param_grid,                              # the values declared above
    cv=folds,                                # the shared canonical folds, never rebuilt
    scoring="neg_mean_absolute_error",       # MAE(z) with a minus sign, because GridSearchCV keeps the highest score
    return_train_score=True,                 # also keep training error, to compare it with validation
)

start = time.time()                                        # time before the search
search.fit(X, z)                                           # all fits on the folds, then one final fit on all rows with the winner 
search_seconds = time.time() - start                       # seconds the whole search took
print("Grid search time (s):", round(search_seconds, 1))

Grid search time (s): 34.5


In [7]:
# TODO: the selected configuration
best = search.best_params_           # dictionary with the winning value of each hyperparameter
best_index = search.best_index_      # row of the winner in the results table

# Chosen value of each hyperparameter, next to the values that were tried, to see whether the winner lies at the edge of the grid
print("max_depth:", best["tree__max_depth"], "| tried:", param_grid["tree__max_depth"])
print("min_samples_leaf:", best["tree__min_samples_leaf"], "| tried:", param_grid["tree__min_samples_leaf"])
print("ccp_alpha:", best["tree__ccp_alpha"], "| tried:", param_grid["tree__ccp_alpha"])

# best_score_ is the winner's validation score averaged over the 5 folds
val_mae = -search.best_score_                                      # undo the minus sign: positive MAE(z)
# cv_results_ is a table with one position per combination; we read the winner's mean training score
train_mae = -search.cv_results_["mean_train_score"][best_index]    # training MAE(z) of the same configuration
print("Mean validation MAE(z):", round(val_mae, 4))
print("Mean training MAE(z):", round(train_mae, 4))

max_depth: 12 | tried: [4, 6, 8, 10, 12, None]
min_samples_leaf: 5 | tried: [5, 10, 20, 40]
ccp_alpha: 0.0 | tried: [0.0, 1e-05, 0.0001, 0.001]
Mean validation MAE(z): 0.1306
Mean training MAE(z): 0.0324


In [8]:
# TODO: MAE(z) for each of the five folds, its mean and spread, RMSE(z), R2(z)
# and one MAE in nanoseconds computed after 10 ** z_hat

# Refit the selected tree on each fold by hand, to get its predictions and compute every metric
# Empty lists, filled with one value per fold inside the loop 
fold_mae = []        # MAE(z) of each fold
fold_rmse = []       # RMSE(z) of each fold
fold_r2 = []         # R2(z) of each fold
fold_mae_ns = []     # MAE in nanoseconds of each fold
fold_seconds = []    # fitting time of each fold

for f in range(5):
    # Rows of this fold 
    train_idx = folds[f][0]         # training rows of fold f
    val_idx = folds[f][1]           # validation rows of fold f

    # Same model as the winner of the search
    model = make_tree()             # fresh, unfitted pipeline
    model.set_params(               # copy the three winning values
        tree__max_depth=best["tree__max_depth"],
        tree__min_samples_leaf=best["tree__min_samples_leaf"],
        tree__ccp_alpha=best["tree__ccp_alpha"],
    )
    # Fit, timing only the fit
    start = time.perf_counter()      # clock starts before fitting
    model.fit(X.iloc[train_idx], z.iloc[train_idx])    # learn only from the training rows
    fold_seconds.append(time.perf_counter() - start)   # seconds this fit took

    # Predict the validation rows 
    z_hat = model.predict(X.iloc[val_idx])            # predicted z for the validation rows
    z_true = z.iloc[val_idx]                          # real z of the same rows

    # Metrics on the z scale
    mae = mean_absolute_error(z_true, z_hat)            # average absolute error, in log10(ns)
    rmse = np.sqrt(mean_squared_error(z_true, z_hat))   # square root of the average squared error
    r2 = r2_score(z_true, z_hat)                        # share of the variance of z that is explained
    # One metric on the physical scale 
    latency_hat = 10 ** z_hat                           # undo the log10: predictions back to nanoseconds
    mae_ns = mean_absolute_error(latency_ns.iloc[val_idx], latency_hat)  # average absolute error, in ns

    # Store the values of this fold
    fold_mae.append(mae)
    fold_rmse.append(rmse)
    fold_r2.append(r2)
    fold_mae_ns.append(mae_ns)

    # Check: GridSearchCV stored the same MAE(z) for this fold (with a minus sign)
    grid_mae = -search.cv_results_["split" + str(f) + "_test_score"][best_index]

    print("fold", f,
         "| MAE(z):", round(mae, 4),
         "| RMSE(z):", round(rmse, 4),
         "| R2(z):", round(r2, 4),
         "| MAE(ns):", round(mae_ns),
         "| MAE(z) from GridSearchCV:", round(grid_mae, 4))

# Lists to arrays, so we can take means here and substract the P2 and P3 scores in P4.5
fold_mae = np.array(fold_mae)
fold_rmse = np.array(fold_rmse)
fold_r2 = np.array(fold_r2)
fold_mae_ns = np.array(fold_mae_ns)

# Summary over the five folds 
mae_spread = fold_mae.max() - fold_mae.min()     # spread as R4 defines it: largest minus smallest fold
print("MAE(z) mean:", round(fold_mae.mean(), 4), "| spread:", round(mae_spread, 4),
     "| min:", round(fold_mae.min(), 4), "| max:", round(fold_mae.max(), 4))
print("RMSE(z) mean:", round(fold_rmse.mean(), 4))
print("R2(z) mean:", round(fold_r2.mean(), 4))
print("MAE(ns) mean:", round(fold_mae_ns.mean()))
# An error of MAE(z) in log10 is a factor of 10 ** MAE(z); minus 1 gives it as a relative error
print("Relative error for MAE(z):", round(10 ** fold_mae.mean() - 1, 2))
print("Mean fitting time of one tree (s):", round(np.mean(fold_seconds), 3))
print("Tree MAE(z) per fold (for R4):", np.round(fold_mae, 4))

fold 0 | MAE(z): 0.1238 | RMSE(z): 0.1882 | R2(z): 0.9503 | MAE(ns): 18018976 | MAE(z) from GridSearchCV: 0.1238
fold 1 | MAE(z): 0.1286 | RMSE(z): 0.2066 | R2(z): 0.94 | MAE(ns): 15327898 | MAE(z) from GridSearchCV: 0.1286
fold 2 | MAE(z): 0.1319 | RMSE(z): 0.2092 | R2(z): 0.9369 | MAE(ns): 12714323 | MAE(z) from GridSearchCV: 0.1319
fold 3 | MAE(z): 0.1353 | RMSE(z): 0.2107 | R2(z): 0.935 | MAE(ns): 12485833 | MAE(z) from GridSearchCV: 0.1353
fold 4 | MAE(z): 0.1332 | RMSE(z): 0.2083 | R2(z): 0.9402 | MAE(ns): 11821343 | MAE(z) from GridSearchCV: 0.1332
MAE(z) mean: 0.1306 | spread: 0.0115 | min: 0.1238 | max: 0.1353
RMSE(z) mean: 0.2046
R2(z) mean: 0.9405
MAE(ns) mean: 14073675
Relative error for MAE(z): 0.35
Mean fitting time of one tree (s): 0.059
Tree MAE(z) per fold (for R4): [0.1238 0.1286 0.1319 0.1353 0.1332]


**Interpretation.** 

*Selected configuration:* The search took 34.5s and chose `max_depth` = 12, `min_samples_leaf` = 5 and `ccp_alpha` = 0.0. Two of these values lie at the edge of the grid, but they are natural limits, not a sign that the grid is too short: no `min_samples_leaf` below 5 gives a different tree, and `ccp_alpha` cannot be lower than 0.0. `max_depth` = 12 lies inside the grid, next to `None`, so the selected tree is limited only by its depth. The mean validation MAE(z) is 0.1306 and the mean training MAE(z) is 0.0324: the tree fits the configurations it was trained on much more closely than new ones.

*Error on the five folds:* The validation MAE(z) goes from 0.1238 to 0.1353, with a mean of 0.1306 and a spread of 0.0115, so it changes little with the configurations held out. Since | z - z_hat| = |log10(predicted/measured)|, a MAE(z) of 0.1306 means that a typical prediction is 10 ** 0.1306 times too high or too low; when it is too high, that is about 35% above the measured latency (relative error 0.35). This is a factor, not an error in nanoseconds. RMSE(z) = 0.2046 is clearly larger than MAE(z), so some configurations have much larger errors than the typical one. R2(z) = 0.9405: the tree explains most of the variation of z.

*Error in nanoseconds:* After 10 ** z_hat the mean MAE is 14073675 ns. Absolute errors grow with the latency itself, so this number is dominated by the slowest configurations, as in Practice 2. The two metrics can therefore order the folds differently: fold 0 has the largest MAE in ns (18018976) and the smallest MAE(z) (0.1238). MAE(z) stays the primary metric because the same relative error costs the same at every latency level.

**What would have gone wrong had the classification hyperparameters been reused.** 

- *Sample size:* cyber has 34928 rows and 34883 feature groups, so almost every row is a different pattern. Storage has 12190 rows but only 2438 different configurations. In cyber a leaf of 5 rows holds about 5 different patterns; in storage it holds only 1 configuration. The same `min_samples_leaf` therefore gives storage leaves with far fewer independent observations.
- *Structure of the rows:* storage rows come in groups of 5 identical replicates. Any `min_samples_leaf` below 5 does nothing here, and leaf sizes must be read in configurations, not rows.
- *Impurity minimised:* the classification tree minimises a class impurity, which measures how mixed the classes are in a node. The regression tree minimises the squared error of z, whose scale is set by the variance of z, 0.7056. A depth that is enough to separate classes need not be enough to follow a continuous latency.
- *Units of ccp_alpha:* in classification ccp_alpha is measured in units of class impurity, here it is in squared units of z. The same number means a different amount of pruning in each task.

# P4.4 — Rules

In [9]:
# TODO: say which fitted tree the rules come from (the selected one, or a shallower tree for reading only)

# The rules come from the selected tree itself: GridSearchCV refitted it on all the development rows
final_pipe = search.best_estimator_                 # preprocessing + tree, already fitted
final_tree = final_pipe.named_steps["tree"]         # the DecisionTreeRegressor
feature_names = final_pipe.named_steps["preprocessing"].get_feature_names_out()   # names of the 17 columns
X_all_t = final_pipe.named_steps["preprocessing"].transform(X)       # the rows as the tree sees them
leaf_of_row = final_tree.apply(X_all_t)             # leaf where each row ends

print("Rules come from: the selected tree (best_estimator_), not a shallower tree")
print("Rows used to fit it:", final_tree.tree_.n_node_samples[0])    # node 0 is the root: it holds every row
print("Depth:", final_tree.get_depth(), "| leaves:", final_tree.get_n_leaves())

Rules come from: the selected tree (best_estimator_), not a shallower tree
Rows used to fit it: 12190
Depth: 12 | leaves: 1456


In [11]:
# TODO: the full rule set of two named leaves, each with its node, predicted value
# and number of training rows (required figure)

tree = final_tree.tree_         # arrays that describe every node of the fitted tree
MIN_ROWS = 20                   # only leaves with at least 4 configurations

# Leaves with at least MIN_ROWS training rows, and the fastest and slowest of them
big_leaves = np.flatnonzero((tree.children_left == -1) & (tree.n_node_samples >= MIN_ROWS))
leaf_z = tree.value[big_leaves, 0, 0]     # predicted z of each of those leaves
fast_leaf = big_leaves[np.argmin(leaf_z)]
slow_leaf = big_leaves[np.argmax(leaf_z)]
print("Leaves with at least", MIN_ROWS, "rows:", len(big_leaves))
print()

def print_rule(leaf, name):
    x = X_all_t[leaf_of_row == leaf][0]     # any training row that ends in this leaf
    print(name, "| node", leaf)
    node = 0      # start at the root
    while tree.children_left[node] != -1:
        column = feature_names[tree.feature[node]].split("__")[-1]  # drop the prefix, whatever it is
        goes_left = x[tree.feature[node]] <= tree.threshold[node]
        if column in NUMERIC:
            threshold = round(tree.threshold[node],3)
            if goes_left:
                print("    ", column, "<=", threshold)
            else:
                print("    ", column, ">", threshold)
        else:
            # one-hot column: "<= 0.5" means the row does NOT have this level
            for nominal in NOMINAL:
                if column.startswith(nominal + "_"):
                    level = column[len(nominal) + 1:]
                    if goes_left:
                        print("    ", nominal, "is not", level)
                    else:
                        print("    ", nominal, "is", level)
        if goes_left:
            node = tree.children_left[node]
        else:
            node = tree.children_right[node]
    assert node == leaf # the manual walk must end where apply() put the row
    z_leaf = tree.value[leaf][0][0]
    print("   predicted z:", round(z_leaf, 4), "| latency = 10 ** z:", round(10 ** z_leaf), "ns")
    print("   training rows:", tree.n_node_samples[leaf], "| configurations:", tree.n_node_samples[leaf] // 5)
    print()

print_rule(fast_leaf, "Leaf A (fastest leaf with at least " + str(MIN_ROWS) + " rows)")
print_rule(slow_leaf, "Leaf B (slowest leaf with at least " + str(MIN_ROWS) + " rows)")

Leaves with at least 20 rows: 119

Leaf A (fastest leaf with at least 20 rows) | node 78
     load_type is not sequential
     n_jobs <= 5.0
     iodepth <= 16.5
     block_size <= 48.0
     raid is not 4+1
     io_type is write
     iodepth > 5.5
     block_size <= 24.0
     read_fraction > 21.5
     iodepth > 9.0
     block_size <= 12.0
     n_disks <= 19.0
   predicted z: 5.3596 | latency = 10 ** z: 228877 ns
   training rows: 30 | configurations: 6

Leaf B (slowest leaf with at least 20 rows) | node 2214
     load_type is sequential
     device_type is not nvme
     n_jobs > 7.5
     iodepth > 12.5
     io_type is write
     n_disks <= 13.0
     iodepth <= 30.5
     n_disks > 4.5
     block_size <= 384.0
     iodepth <= 29.0
     raid is not 2+1
     raid is 4+1
   predicted z: 8.6397 | latency = 10 ** z: 436246220 ns
   training rows: 20 | configurations: 4



**Which tree and which leaves.** The rules come from the selected tree itself (`best_estimator_`), which `GridSearchCV` refitted on all 12190 development rows; no shallower tree was fitted just for reading. It has depth 12 and 1456 leaves, far too many to read as a whole, so we show the complete rule of two leaves. To avoid leaves that describe a single configuration, we only consider leaves with at least 20 training rows (4 configurations). This reading rule was fixed before looking at the leaves and does not depend on any score; 119 leaves pass it. Leaf A has the lowest predicted z and Leaf B the highest, so they show the two extremes of the tree, not typical leaves.

**How to read the figure.** Each line is one test on the path from the root to the leaf, in order. Numeric thresholds lie halfway between two observed values. For a one-hot column, the tree tests whether the level is present, printed as "is" or "is not". Some variables are tested more than once with a tighter bound each time (`block_size <= 48.0`, `<= 24.0`, `<=12.0` in Leaf A); only the tightest one matters. The predicted z of a leaf is the mean z of its training rows, so 10 ** z is the geometric mean of their latencies.

- **Leaf A** (node 78, 30 training rows, 6 configurations): random access (`load_type` is not sequential), write, a RAID scheme other than 4+1, `n_jobs <= 5.0`, `iodepth` above 9.0 and at most 16.5, `block_size <= 12.0`, `read_fraction > 21.5` and `n_disks <= 19.0`. Predicted latency: 228877 ns.

**One rule read as an operations statement (Leaf B: node 2214, 20 training rows, 4 configurations).** On an hdd array in RAID 4+1 with more than 4.5 and at most 13.0 disks, a sequential write workload with more than 7.5 jobs, a queue depth above 12.5 and at most 29.0, and a block size up to 384.0 is predicted to have a latency of about 436246220 ns, just under half a second. That is more than three orders of magnitude above Leaf A(z = 5.3596 against z = 8.6397).

This statement summarises only 4 training configurations. It describes an association in the fitted tree, not a cause, and it only holds inside the ranges seen in training. A leaf predicts the same value for every configuration it contains, so the rule cannot say how latency changes within these ranges.

# P4.5 — Challenge B against Practice 3 and Practice 2

In [12]:
# TODO: the paired per-fold differences against the Practice 3 linear regression and the Practice 2 KNN

# Five per-fold MAE(z) of one earlier model, read from comparativa.csv
def previous_scores(milestone, challenge, family):
    table = pd.read_csv("../../comparativa.csv")
    # keep only the row with this milestone, challenge and family
    row = table[(table["milestone"] == milestone) & (table["challenge"] == challenge) & (table["family"] == family)]
    # the five scores are stored as one text
    text = row["primary_metric_per_fold"].iloc[0]
    # cut the text at the spaces and turn each piece into a number
    return np.array(text.split(), dtype=float)

p3_storage = previous_scores("P3", "storage", "LinearRegression")
p2_storage = previous_scores("P2", "storage", "KNN")
tree_storage = np.round(fold_mae, 4)  # same 4 decimals as the stored scores

# MAE(z): lower is better, so delta = previous - tree (positive means the tree did better)
delta_p3 = p3_storage - tree_storage
delta_p2 = p2_storage - tree_storage

print("Tree:", tree_storage)
print("P3 linear:", p3_storage)
print("P2 KNN:", p2_storage)
print("P3 - tree:", np.round(delta_p3, 4),
     "| mean:", round(delta_p3.mean(), 4),
     "| spread:", round(delta_p3.max() - delta_p3.min(), 4),
     "| folds where the tree is better:", int((delta_p3 > 0).sum()), "of 5")
print("P2 - tree:", np.round(delta_p2, 4),
     "| mean:", round(delta_p2.mean(), 4),
     "| spread:", round(delta_p2.max() - delta_p2.min(), 4),
     "| folds where the tree is better:", int((delta_p2 > 0).sum()), "of 5")

Tree: [0.1238 0.1286 0.1319 0.1353 0.1332]
P3 linear: [0.2047 0.1937 0.1984 0.1953 0.1987]
P2 KNN: [0.1816 0.187  0.1778 0.1775 0.1762]
P3 - tree: [0.0809 0.0651 0.0665 0.06   0.0655] | mean: 0.0676 | spread: 0.0209 | folds where the tree is better: 5 of 5
P2 - tree: [0.0578 0.0584 0.0459 0.0422 0.043 ] | mean: 0.0495 | spread: 0.0162 | folds where the tree is better: 5 of 5


In [13]:
# Numbers of the earlier storage models, read from comparativa.csv, used in the answers below
table = pd.read_csv("../../comparativa.csv")
storage_rows = table[table["challenge"] == "storage"]
for i in range(len(storage_rows)):
    row = storage_rows.iloc[i]
    print(row["milestone"], row["family"],
         "| MAE(z):", row["primary_metric"],
         "| spread:", row["primary_metric_spread"],
         "| fit time (s):", row["fit_time_s"])
    print("     ", row["supporting_metrics"])

# Share of the Dummy MAE(z) that each model removes (1 = perfect, 0 = no better than the Dummy)
dummy_mae = previous_scores("P2", "storage", "Dummy").mean()
print("Share of the Dummy MAE(z) removed | tree:", round(1 - fold_mae.mean() / dummy_mae, 3),
     "| P2 KNN:", round(1 - p2_storage.mean() / dummy_mae, 3),
     "| P3 linear:", round(1 - p3_storage.mean() / dummy_mae, 3))

P2 Dummy | MAE(z): 0.677 | spread: 0.0294 | fit time (s): 0.003
      RMSE(z) 0.8400; R2(z) -0.0001; MAE_ns 27037511 (computed after inverting predictions)
P2 KNN | MAE(z): 0.18 | spread: 0.0109 | fit time (s): 0.0127
      RMSE(z) 0.2568; R2(z) 0.9065; MAE_ns 17175412 (computed after inverting predictions)
P3 LinearRegression | MAE(z): 0.1981 | spread: 0.011 | fit time (s): 0.0298
      RMSE(z) 0.2703; R2(z) 0.8964; MAE_ns 16289769 (computed after inverting predictions)
Share of the Dummy MAE(z) removed | tree: 0.807 | P2 KNN: 0.734 | P3 linear: 0.707


### The six fixed comparison questions

Answer each one for both comparisons: the tree against the Practice 3 linear regression and against the Practice 2 KNN.

1. **Size and consistency of the change.** Against the Practice 3 linear regression, the tree has a lower MAE(z) on all 5 folds: differences 0.0809, 0.0651, 0.0665, 0.06 and 0.0655, mean 0.0676, spread 0.0209. Against the Practice 2 KNN it is also better on all 5 folds: differences 0.0578, 0.0584, 0.0459, 0.0422 and 0.043, mean 0.0495, spread 0.0162; the gain is smaller on the last three folds. Even the smallest difference (0.06 against the linear model, 0.0422 against KNN) is several times the fold-to-fold spread of each model (0.0115 for the tree, 0.011 for the linear model, 0.0109 for KNN). This only describes these five folds; it is not a statistical test. The tree and the KNN were both tuned on these same folds, so both scores are slightly optimistic, the tree probably a bit more because its search was larger.
2. **Route and representation.** All three models use the primary route, the same 9 source predictors, the same target z and the same canonical folds, but not the same representation: Practice 2 uses full one-hot with `StandardScaler`, Practice 3 drops one reference level per nominal predictor and also scales, and Practice 4 uses full one-hot with unscaled numeric columns. Strictly, both comparisons are pipeline comparisons.

   - Against Practice 2, the only difference is the scaler. Rescaling each column with a positive factor keeps the order of its values, so it keeps the partition that every threshold induces: fitted on the scaled columns, the tree would be the same tree, apart from the threshold numbers (and tie-breaking and finite precision). We did not refit it to check. This comparison is therefore close to a family comparison: on the same encoded columns, the tree did better than KNN on these folds.
   - Against Practice 3, the encoding also differs. With the reference level dropped, a test such as "raid is 4+1" (used in Leaf B) is not a single column and need several splits. The evidence supports "this tree pipeline did better than that linear pipeline", not that the family alone caused the whole difference.

3. **What can the new family express?** Against the linear model: interactions and thresholds. The linear model on z adds a fixed amount per predictor, whatever the other predictors are. The tree's prediction depends on combinations (Leaf B only applies when sequential access, hdd, write and RAID 4+1 hold together), and it can place a threshold where latency jumps. Against KNN: KNN can also follow interactions locally, but its distance gives the same weight to every standardised predictor and it always averages the same number of neighbours. The tree chooses which predictors matter in each region, ignores the others there, and gives explicit rules. What the tree cannot express is a smooth change: its prediction is piecewise constant, with at most 1456 different values, and it stays constant outside the ranges seen in training.
4. **What did it cost?** Fitting time is small for all three: 0.059 s per fold for the tree, against 0.0298 s for the linear model and 0.0127 s for KNN. These were measured in different runs, so only the order of magnitude matters. Tuning effort is clearly higher: 96 combinations and 480 fits, 34.5 s in total, against no search for the linear model (Ridge and Lasso were explored separately in Practice 3) and a one-dimensional grid of K for KNN. Interpretability improves against KNN, which gives no rules at all. Against the linear model the change is mixed: each path can be read, but it has 12 conditions, and the whole tree, with 1456 leaves, cannot be read at once, while the linear model is summarised by one coefficient per column.
5. **Did the ordering of the two challenges' difficulty change** In storage, the share of the Dummy MAE(z) removed is 0.807 for the tree, 0.734 for KNN and 0.707 for the linear model. TODO (with R2's cyber tree): in cyber the tree has a macro F1 of ..., which closes ... of the gap between the Dummy and a perfect score, against ... for KNN and ... for logistic regression. Conclusion: ...
6. **Would you ship the new model?** Yes, as the Challenge B model, replacing the KNN. It has the lowest MAE(z) on all five folds, and, unlike in Practice 3, no supporting metric disagrees: RMSE(z) 0.2046 against 0.2568 (KNN) and 0.2703 (linear), R2(z) 0.9405 against 0.9065 and 0.8964, and MAE in ns 14073675 against 17175412 and 16289769. Caveats: the gap between training and validation MAE(z) is large (0.0324 against 0.1306), the configuration was chosen on the same folds that measure it, a single tree can change a lot with small changes in the data, and it cannot extrapolate beyond the training ranges. We would change our mind if the result on the final test were clearly worse, if a family from a later practice did better on these folds, or if we needed predictions for configurations outside the ranges seen in training.